# Algorithmic Trading and Quantitative Strategies
## Part 7: Trend Following Strategies with Backtrader
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. Trend Following Fundamentals
2. Moving Average Systems
3. MACD System
4. Donchian Channel Breakout
5. Bollinger Band Breakout
6. Time Series Momentum
7. Adaptive Moving Averages
8. Strategy Comparison
9. Exercises

## 1. Trend Following Fundamentals

Trend following is one of the oldest and most widely practiced systematic trading strategies. The core philosophy is simple:

> **"Cut your losses short and let your profits run."**

### 1.1 Three Statistical Regimes

Financial time series can be in one of three regimes:
1. **Trending (positive autocorrelation):** Price movements persist — a rise is likely followed by another rise
2. **Mean-reverting (negative autocorrelation):** Price movements reverse — a rise is likely followed by a decline
3. **Random walk (zero autocorrelation):** Past movements provide no information about future movements

Trend following strategies are profitable in regime 1 and lose money in regime 3 and especially regime 2.

### 1.2 Why Does Trend Following Work?

Behavioral explanations:
- **Under-reaction:** Investors are slow to incorporate new information
- **Over-reaction:** Once a trend is recognized, herding amplifies it
- **Anchoring:** Investors anchor to past prices and are slow to update
- **Disposition effect:** Sell winners too early, hold losers too long

Risk-based explanations:
- Trend following is a long-volatility strategy (like owning options)
- Performs well in crisis periods ("crisis alpha")
- The **trend following smile:** positive returns in both extreme up and down markets

### 1.3 The Trend Following Smile

Historical analysis shows that trend following strategies tend to have a convex payoff profile:
- Large profits in strongly trending markets (both up and down)
- Small losses in choppy/range-bound markets
- This provides natural **portfolio insurance** — diversification benefit when combined with traditional long-only portfolios

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import backtrader as bt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)

# Download data for examples
spy_data = yf.download('SPY', start='2010-01-01', end='2024-12-31')
spy_data.columns = spy_data.columns.droplevel('Ticker')
print(f"Downloaded {len(spy_data)} bars for SPY")

## 2. Moving Average Systems

### 2.1 Single Moving Average Crossover

The simplest trend following system:
- **Buy** when price crosses above the moving average
- **Sell** when price crosses below the moving average

Variants: SMA (Simple), EMA (Exponential), WMA (Weighted)

In [ ]:
class SingleMACrossover(bt.Strategy):
    """Price vs Single Moving Average crossover."""
    params = dict(
        period=50,
        ma_type='SMA',  # 'SMA', 'EMA', 'WMA'
    )
    
    def __init__(self):
        if self.p.ma_type == 'SMA':
            self.ma = bt.ind.SMA(self.data.close, period=self.p.period)
        elif self.p.ma_type == 'EMA':
            self.ma = bt.ind.EMA(self.data.close, period=self.p.period)
        elif self.p.ma_type == 'WMA':
            self.ma = bt.ind.WMA(self.data.close, period=self.p.period)
        
        self.crossover = bt.ind.CrossOver(self.data.close, self.ma)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()

### 2.2 Double Moving Average Crossover

Uses two moving averages — a fast and a slow:
- **Buy** when fast MA crosses above slow MA (Golden Cross)
- **Sell** when fast MA crosses below slow MA (Death Cross)

This is smoother and generates fewer signals than single MA systems.

In [ ]:
class DoubleMACrossover(bt.Strategy):
    """Double Moving Average Crossover."""
    params = dict(
        fast_period=50,
        slow_period=200,
        ma_type='SMA',
    )
    
    def __init__(self):
        ma_class = {'SMA': bt.ind.SMA, 'EMA': bt.ind.EMA, 'WMA': bt.ind.WMA}[self.p.ma_type]
        self.fast_ma = ma_class(self.data.close, period=self.p.fast_period)
        self.slow_ma = ma_class(self.data.close, period=self.p.slow_period)
        self.crossover = bt.ind.CrossOver(self.fast_ma, self.slow_ma)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()

## 3. MACD System

The **MACD (Moving Average Convergence Divergence)** is a momentum/trend indicator:

$$\text{MACD} = \text{EMA}(\text{fast}) - \text{EMA}(\text{slow})$$
$$\text{Signal} = \text{EMA}(\text{signal period}) \text{ of MACD}$$
$$\text{Histogram} = \text{MACD} - \text{Signal}$$

Trading rules:
- **Buy** when MACD crosses above Signal line
- **Sell** when MACD crosses below Signal line

In [ ]:
class MACDTrendStrategy(bt.Strategy):
    """MACD trend following strategy."""
    params = dict(
        fast=12,
        slow=26,
        signal=9,
    )
    
    def __init__(self):
        self.macd = bt.ind.MACD(
            self.data.close,
            period_me1=self.p.fast,
            period_me2=self.p.slow,
            period_signal=self.p.signal,
        )
        self.crossover = bt.ind.CrossOver(self.macd.macd, self.macd.signal)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()

## 4. Donchian Channel Breakout

The **Donchian Channel** is formed by the highest high and lowest low over $n$ periods:

$$\text{Upper} = \max(H_{t}, H_{t-1}, \ldots, H_{t-n+1})$$
$$\text{Lower} = \min(L_{t}, L_{t-1}, \ldots, L_{t-n+1})$$
$$\text{Middle} = \frac{\text{Upper} + \text{Lower}}{2}$$

This was the basis of the famous **Turtle Trading** system:
- **Buy** when price breaks above the upper channel (new 20-day high)
- **Sell** when price breaks below the lower channel of a shorter period (new 10-day low)

In [ ]:
class DonchianChannel(bt.Indicator):
    """Custom Donchian Channel indicator."""
    lines = ('upper', 'lower', 'middle',)
    params = dict(period=20,)
    
    def __init__(self):
        self.lines.upper = bt.ind.Highest(self.data.high, period=self.p.period)
        self.lines.lower = bt.ind.Lowest(self.data.low, period=self.p.period)
        self.lines.middle = (self.lines.upper + self.lines.lower) / 2.0


class DonchianBreakout(bt.Strategy):
    """Donchian Channel Breakout (Turtle-style)."""
    params = dict(
        entry_period=20,
        exit_period=10,
    )
    
    def __init__(self):
        self.entry_channel = DonchianChannel(self.data, period=self.p.entry_period)
        self.exit_channel = DonchianChannel(self.data, period=self.p.exit_period)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.data.close[0] > self.entry_channel.upper[-1]:
                self.order = self.buy()
        else:
            if self.data.close[0] < self.exit_channel.lower[-1]:
                self.order = self.close()

## 5. Bollinger Band Breakout

Bollinger Bands measure price relative to volatility:

$$\text{Upper} = \text{SMA}(n) + k \cdot \sigma(n)$$
$$\text{Lower} = \text{SMA}(n) - k \cdot \sigma(n)$$

Breakout strategy (trend following, not mean reversion):
- **Buy** when price closes above the upper band
- **Sell** when price closes below the middle band (SMA)

In [ ]:
class BollingerBreakout(bt.Strategy):
    """Bollinger Band Breakout strategy (trend following)."""
    params = dict(
        period=20,
        devfactor=2.0,
    )
    
    def __init__(self):
        self.boll = bt.ind.BollingerBands(
            self.data.close, period=self.p.period, devfactor=self.p.devfactor)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.data.close[0] > self.boll.top[0]:
                self.order = self.buy()
        else:
            if self.data.close[0] < self.boll.mid[0]:
                self.order = self.close()

## 6. Time Series Momentum

Time series momentum (TSMOM) is a simple trend signal based on past returns:

$$\text{Signal}_t = \text{sign}(r_{t-1, t-n})$$

where $r_{t-1, t-n}$ is the return over the past $n$ periods.

- **Buy** if the past $n$-day return is positive
- **Sell/Flat** if the past $n$-day return is negative

Risk-adjusted version:

$$\text{Signal}_t = \text{sign}\left(\frac{r_{t-1, t-n}}{\sigma_t}\right)$$

In [ ]:
class TimeSeriesMomentum(bt.Strategy):
    """Time Series Momentum strategy."""
    params = dict(
        lookback=126,  # ~6 months
        vol_lookback=20,
        risk_adjusted=True,
    )
    
    def __init__(self):
        self.returns = bt.ind.PctChange(self.data.close, period=self.p.lookback)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        
        if len(self) < self.p.lookback + 1:
            return
        
        momentum = self.returns[0]
        
        if not self.position:
            if momentum > 0:
                self.order = self.buy()
        else:
            if momentum <= 0:
                self.order = self.close()

## 7. Adaptive Moving Averages

### 7.1 KAMA (Kaufman Adaptive Moving Average)

KAMA adapts its smoothing speed based on market conditions:

$$\text{ER} = \frac{|\text{Direction}|}{\text{Volatility}} = \frac{|P_t - P_{t-n}|}{\sum_{i=0}^{n-1} |P_{t-i} - P_{t-i-1}|}$$

$$\text{SC} = (\text{ER} \cdot (\text{fast} - \text{slow}) + \text{slow})^2$$

$$\text{KAMA}_t = \text{KAMA}_{t-1} + \text{SC} \cdot (P_t - \text{KAMA}_{t-1})$$

When the market is trending strongly (ER close to 1), KAMA responds quickly. In choppy markets (ER close to 0), KAMA moves slowly.

In [ ]:
class KAMA(bt.Indicator):
    """Kaufman Adaptive Moving Average."""
    lines = ('kama',)
    params = dict(
        period=10,
        fast=2,
        slow=30,
    )
    
    def __init__(self):
        direction = abs(self.data - self.data(-self.p.period))
        volatility = bt.ind.SumN(abs(self.data - self.data(-1)), period=self.p.period)
        
        er = direction / (volatility + 1e-10)
        
        fast_sc = 2.0 / (self.p.fast + 1.0)
        slow_sc = 2.0 / (self.p.slow + 1.0)
        sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2
        
        self.lines.kama = bt.ind.ExponentialSmoothing(self.data, period=1, alpha=sc)


class KAMAStrategy(bt.Strategy):
    """KAMA trend following strategy."""
    params = dict(period=10, fast=2, slow=30)
    
    def __init__(self):
        self.kama = KAMA(self.data.close, 
                        period=self.p.period, fast=self.p.fast, slow=self.p.slow)
        self.crossover = bt.ind.CrossOver(self.data.close, self.kama)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()

## 8. Strategy Comparison

Let us run all trend following strategies on the same data and compare their performance.

In [ ]:
def run_strategy(strategy_class, data_df, cash=100000, commission=0.001, **kwargs):
    """Run a strategy and return performance metrics."""
    cerebro = bt.Cerebro()
    cerebro.adddata(bt.feeds.PandasData(dataname=data_df))
    cerebro.addstrategy(strategy_class, **kwargs)
    cerebro.broker.setcash(cash)
    cerebro.broker.setcommission(commission=commission)
    cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
    
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.0)
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    
    results = cerebro.run()
    strat = results[0]
    
    sharpe = strat.analyzers.sharpe.get_analysis()
    dd = strat.analyzers.drawdown.get_analysis()
    ta = strat.analyzers.trades.get_analysis()
    ret = strat.analyzers.returns.get_analysis()
    
    total_trades = ta.get('total', {}).get('total', 0)
    won = ta.get('won', {}).get('total', 0)
    
    return {
        'Total Return': f"{ret.get('rtot', 0):.2%}",
        'Ann. Return': f"{ret.get('rnorm', 0):.2%}",
        'Sharpe': f"{sharpe.get('sharperatio', 0) or 0:.3f}",
        'Max DD': f"{dd.max.drawdown:.1f}%",
        'Trades': total_trades,
        'Win Rate': f"{won/total_trades:.1%}" if total_trades > 0 else 'N/A',
        'Final Value': f"${cerebro.broker.getvalue():,.0f}",
    }

# Define strategies to compare
strategies = {
    'SMA(50)': (SingleMACrossover, {'period': 50, 'ma_type': 'SMA'}),
    'EMA(50)': (SingleMACrossover, {'period': 50, 'ma_type': 'EMA'}),
    'Double SMA(50/200)': (DoubleMACrossover, {'fast_period': 50, 'slow_period': 200}),
    'Double EMA(20/50)': (DoubleMACrossover, {'fast_period': 20, 'slow_period': 50, 'ma_type': 'EMA'}),
    'MACD(12/26/9)': (MACDTrendStrategy, {'fast': 12, 'slow': 26, 'signal': 9}),
    'Donchian(20/10)': (DonchianBreakout, {'entry_period': 20, 'exit_period': 10}),
    'Bollinger Breakout': (BollingerBreakout, {'period': 20, 'devfactor': 2.0}),
    'TSMOM(126)': (TimeSeriesMomentum, {'lookback': 126}),
    'TSMOM(252)': (TimeSeriesMomentum, {'lookback': 252}),
}

# Run all strategies
comparison = {}
for name, (strat_class, params) in strategies.items():
    try:
        comparison[name] = run_strategy(strat_class, spy_data, **params)
        print(f"\u2713 {name}")
    except Exception as e:
        print(f"\u2717 {name}: {e}")

# Display results
comp_df = pd.DataFrame(comparison).T
print("\n" + "=" * 80)
print("TREND FOLLOWING STRATEGY COMPARISON (SPY 2010-2024)")
print("=" * 80)
print(comp_df.to_string())

### 8.1 Equivalence of Trend Following Systems

Levine & Pedersen (2016) showed that many trend following systems are mathematically related:

- All linear trend indicators are weighted averages of past price changes
- SMA crossover, MACD, and other MA-based systems differ mainly in how they weight recent vs. distant observations
- The choice of trend indicator matters less than the choice of **lookback period** and **risk management**

### 8.2 Key Insights from the Comparison

1. **All trend following strategies are profitable on SPY over long horizons** — this is consistent with the positive drift in equity markets
2. **Shorter-period systems trade more frequently** but may be more susceptible to whipsaws
3. **Longer-period systems capture larger trends** but enter/exit late
4. **Risk management (stops, sizing) matters at least as much as signal generation**

---
### References
- Hurst, Ooi, & Pedersen (2017). *A Century of Evidence on Trend-Following Investing.* AQR.
- Levine & Pedersen (2016). *Which Trend Is Your Friend?* Financial Analysts Journal.
- Antonacci, G. (2014). *Dual Momentum Investing.*
- Covel, M. (2009). *Trend Following.*
- Curtis Faith (2007). *Way of the Turtle.*